In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import zipfile
from scipy.stats import entropy
from multiprocessing import Pool
import time
import torch
import torch.nn as nn
from typing import List, Tuple, Optional

class GaitSTARProcessor:
    def __init__(self, input_size=(128, 128)):
        self.input_size = input_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Initialize feature transformation network
        self.feature_transform = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU()
        ).to(self.device)
        
    def preprocess_frame(self, image: np.ndarray) -> np.ndarray:
        """Enhanced frame preprocessing"""
        if image.size == 0:
            return None
            
        # Resize frame
        image = cv2.resize(image, self.input_size)
        
        # Apply CLAHE for contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(image)
        
        # Edge preservation
        enhanced = cv2.bilateralFilter(enhanced, 9, 75, 75)
        
        return enhanced
        
    def compute_optical_flow(self, frames: List[np.ndarray]) -> np.ndarray:
        """Compute optical flow features"""
        flows = []
        for i in range(len(frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(
                frames[i], frames[i + 1],
                None, 0.5, 3, 15, 3, 5, 1.2, 0
            )
            magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            magnitude = cv2.normalize(magnitude, None, 0, 1, cv2.NORM_MINMAX)
            flow_feature = np.stack([magnitude, angle], axis=-1)
            flows.append(flow_feature)
            
        return np.array(flows)
        
    def compute_gaitstar(self, sequence_images: List[np.ndarray]) -> Optional[np.ndarray]:
        """Compute GaitSTAR features for a sequence"""
        if not sequence_images:
            return None
            
        # Preprocess frames
        processed_frames = []
        for frame in sequence_images:
            processed = self.preprocess_frame(frame)
            if processed is not None:
                processed_frames.append(processed)
                
        if len(processed_frames) < 2:
            return None
            
        # Compute optical flow
        flow_features = self.compute_optical_flow(processed_frames)
        
        # Convert to tensor for feature extraction
        sequence_tensor = torch.FloatTensor(processed_frames).unsqueeze(1).to(self.device)
        
        # Extract CNN features
        cnn_features = []
        with torch.no_grad():
            for i in range(len(sequence_tensor)):
                feat = self.feature_transform(sequence_tensor[i].unsqueeze(0))
                cnn_features.append(feat.cpu().numpy())
        
        cnn_features = np.array(cnn_features)
        
        # Combine features
        combined_features = np.concatenate([
            cnn_features.reshape(len(cnn_features), -1),
            flow_features.reshape(len(flow_features), -1)
        ], axis=1)
        
        # Apply temporal pooling with attention weights
        attention_weights = np.exp(combined_features.mean(axis=1))
        attention_weights = attention_weights / np.sum(attention_weights)
        
        gaitstar_feature = np.sum(combined_features * attention_weights[:, np.newaxis], axis=0)
        
        return gaitstar_feature

def process_subject(args):
    subject_folder, base_folder, output_folder, camera_angles, counter = args
    start_time = time.time()
    
    gaitstar_processor = GaitSTARProcessor()
    
    subject_path = os.path.join(base_folder, subject_folder, subject_folder)
    if not os.path.isdir(subject_path):
        print(f"No inner subject folder found for: {subject_folder}")
        return [], f"{counter}. Finished processing subject: {subject_folder}", 0
    
    metadata = []
    for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
        nm_folder_path = os.path.join(subject_path, nm_folder)
        if not os.path.exists(nm_folder_path):
            print(f"No {nm_folder} folder found for subject: {subject_folder}")
            continue

        for cam_folder in camera_angles:
            cam_path = os.path.join(nm_folder_path, cam_folder)
            if not os.path.isdir(cam_path):
                continue
            
            image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            if not image_files:
                print(f"No images found in {cam_path}")
                continue
            
            sequence_images = []
            for file in image_files:
                image_path = os.path.join(cam_path, file)
                image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                if image is not None and image.size > 0:
                    sequence_images.append(image)
                else:
                    print(f"Failed to load image or empty image: {image_path}")
            
            if sequence_images:
                gaitstar_feature = gaitstar_processor.compute_gaitstar(sequence_images)
                
                if gaitstar_feature is not None and gaitstar_feature.size > 0:
                    # Normalize and convert to image format
                    feature_image = ((gaitstar_feature - gaitstar_feature.min()) / 
                                   (gaitstar_feature.max() - gaitstar_feature.min()) * 255).astype(np.uint8)
                    
                    # Reshape to 2D if necessary
                    sqrt_size = int(np.ceil(np.sqrt(feature_image.shape[0])))
                    padded_size = sqrt_size * sqrt_size
                    padded_feature = np.pad(feature_image, (0, padded_size - len(feature_image)))
                    feature_image_2d = padded_feature.reshape(sqrt_size, sqrt_size)
                    
                    filename = f"{subject_folder}_{nm_folder}_{cam_folder}.png"
                    output_path = os.path.join(output_folder, 'gaitstar_images', filename)
                    cv2.imwrite(output_path, feature_image_2d)
                    
                    metadata.append({
                        'image_filename': filename,
                        'label': int(subject_folder),
                        'cam_angle': int(cam_folder),
                        'nm_sequence': nm_folder
                    })
                else:
                    print(f"Failed to create valid GaitSTAR feature for {cam_path}")
    
    processing_time = time.time() - start_time
    return metadata, f"{counter}. Finished processing subject: {subject_folder} (Time: {processing_time:.2f} seconds)", processing_time

def create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=None):
    total_start_time = time.time()
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    gaitstar_images_folder = os.path.join(output_folder, 'gaitstar_images')
    if not os.path.exists(gaitstar_images_folder):
        os.makedirs(gaitstar_images_folder)
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    process_args = [(subject, base_folder, output_folder, camera_angles, i) 
                    for i, subject in enumerate(subject_folders, 1)]
    
    with Pool() as pool:
        results = pool.map(process_subject, process_args)
    
    all_metadata = [item for result in results for item in result[0]]
    total_processing_time = 0
    for _, message, processing_time in results:
        print(message)
        total_processing_time += processing_time
    
    if all_metadata:
        df = pd.DataFrame(all_metadata)
        csv_path = os.path.join(output_folder, 'gaitstar_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Successfully created {len(df)} GaitSTAR images for {len(df['label'].unique())} subjects.")
        print(f"Metadata saved to: {csv_path}")
    else:
        print("No GaitSTAR images were successfully created.")
    
    total_time = time.time() - total_start_time
    print(f"Total processing time for all subjects: {total_processing_time:.2f} seconds")
    print(f"Total execution time including overhead: {total_time:.2f} seconds")

def zip_dataset(output_folder, zip_filename):
    print(f"Creating zip file: {zip_filename}")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)
    print(f"Zip file created: {zip_filename}")

if __name__ == "__main__":
    base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
    output_folder = '/kaggle/working/gaitstar_dataset'
    zip_filename = '/kaggle/working/gaitstar_processed_dataset.zip'
    camera_angles = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']

    create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=camera_angles)
    zip_dataset(output_folder, zip_filename)
    print(f"GaitSTAR dataset has been created and zipped. You can now download {zip_filename}")